In [81]:
import numpy as np 
import pandas as pd
from astropy.io import fits

In [82]:
input_file = '/Users/giadaaggio/Desktop/Thesis/TOTORO/FITS/Catalogs/catalog.xym'
# load only the columns you need so x, y, F814W and F555W, ingore the header
df = pd.read_csv(input_file, delim_whitespace=True)
df["x"].fillna(df["x"].median(), inplace=True)
df["y"].fillna(df["y"].median(), inplace=True)
df["F814W"].fillna(df["F814W"].median(), inplace=True)
df["F555W"].fillna(df["F555W"].median(), inplace=True)

data = df[['x', 'y', 'F814W', 'F555W']]


In [83]:
print(data)

             x        y   F814W    F555W
0      5894.85  1143.01 -10.457 -10.1313
1      5787.07  1148.84  -7.412 -10.2728
2      5823.93  1157.60  -8.794 -10.2728
3      5832.33  1157.62  -9.674 -10.2728
4      5678.37  1164.50  -7.632 -10.2728
...        ...      ...     ...      ...
95408  8905.37  8929.01 -10.120 -10.2728
95409  8825.94  8930.33  -7.115 -10.2728
95410  8816.49  8931.44  -8.155 -10.2728
95411  8995.33  8931.32  -7.360 -10.2728
95412  9016.47  8935.28  -9.973 -10.2728

[95413 rows x 4 columns]


In [84]:
# create a simple dataframe with the zeropoints for the calibration
zero_F814W = 32.1551
zero_F555W = 32.2380

data['F555W_cal'] = data['F555W'] + zero_F555W
data['F814W_cal'] = data['F814W'] + zero_F814W

/var/folders/nx/ljvhqy816sn3y2d3hzwxcqhc0000gn/T/ipykernel_22599/3806336715.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['F555W_cal'] = data['F555W'] + zero_F555W
/var/folders/nx/ljvhqy816sn3y2d3hzwxcqhc0000gn/T/ipykernel_22599/3806336715.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['F814W_cal'] = data['F814W'] + zero_F814W


In [85]:
X = data['x'].to_numpy()
Y = data['y'].to_numpy()
Mag1 = data['F555W'].to_numpy()
Mag2 = data['F814W'].to_numpy()

In [86]:
print(X, Y, Mag1, Mag2)

[5894.85 5787.07 5823.93 ... 8816.49 8995.33 9016.47] [1143.01 1148.84 1157.6  ... 8931.44 8931.32 8935.28] [-10.1313 -10.2728 -10.2728 ... -10.2728 -10.2728 -10.2728] [-10.457  -7.412  -8.794 ...  -8.155  -7.36   -9.973]


In [87]:
flux1 = 10**(-0.4 * Mag1)
flux2 = 10**(-0.4 * Mag2)

flux_bw = (flux1 + flux2) / 2  # Average the fluxes

In [88]:
# Define the image size (adjust based on the data)
img_size = int(max(X.max(), Y.max())) + 10
image1 = np.zeros((img_size, img_size))
image2 = np.zeros((img_size, img_size))

image_bw = np.zeros((img_size, img_size))

# Populate images with flux values
for x, y, f1, f2 in zip(X.astype(int), Y.astype(int), flux1, flux2):
    image1[y, x] += f1  # Assign flux to correct position
    image2[y, x] += f2

for x, y, f in zip(X.astype(int), Y.astype(int), flux_bw):
    image_bw[y, x] += f

In [89]:
fits.writeto("image1.fits", image1, overwrite=True)
fits.writeto("image2.fits", image2, overwrite=True)
fits.writeto("bw_image.fits", image_bw, overwrite=True)